# Cox Proportional Hazard Analysis using the Rhino SDK

*Notebook Last Validated: 2026-08-06*

## What does this notebook do?

This notebook runs a **Cox proportional hazard** analysis across multiple datasets using the Rhino Federated Computing Platform (FCP).

### What is Cox proportional hazard analysis?

Cox regression is a statistical method for answering questions like:

> *"Which factors affect **how long** it takes for a specific event to happen — and by how much?"*

Common examples in healthcare:
- Does a certain treatment reduce the time until a patient recovers?
- Do factors like age or a lab value predict whether — and how soon — a patient experiences a health event?

For each person in your data, the analysis needs two key pieces of information:
- **Time** — how long until the event occurred (or how long the person was observed if it never did)
- **Event** — whether the event actually happened (`1` = yes, `0` = no / still being observed)

It also accounts for **covariates** — other variables that might influence the outcome, such as clinical measurements or demographics.

The result is a set of coefficients (one per covariate) that describe how strongly each factor is associated with the timing of the event.

### What does "federated" mean here?

In a federated analysis, the raw patient data **never leaves the sites that hold it**. Instead, the Rhino FCP coordinates the computation across sites, and only aggregated statistical results are shared. This lets you run analyses on larger, more diverse populations while preserving data privacy.

### What you'll need

- A Rhino FCP account with access to a project
- Two or more datasets in that project, each containing a time column, an event column, and covariate columns
- The `rhino_health` Python package (see Setup below)

## Setup

Ensure you've installed the `rhino_health` SDK before running this notebook. Python 3.8 or later is required.

In [1]:
pip install rhino-health

Note: you may need to restart the kernel to use updated packages.


## Step 1: Load Libraries

This cell imports the tools this notebook depends on. You don't need to understand what each one does — just run it before anything else.

In [2]:
from getpass import getpass
import rhino_health
import pandas as pd
from rhino_health.lib.metrics import *

## Step 2: Log in to the Rhino FCP

Enter your Rhino FCP credentials to start a session. This is the same email and password you use to log in at the Rhino FCP web portal.

When you run the cell below, a password prompt will appear — your password is never stored or displayed.

In [3]:
my_username = "stephanie@rhinohealth.com" # Replace this with the email you use to log into Rhino Health

print("Logging In")
session = rhino_health.login(username=my_username, password=getpass())
print("Logged In")

Logging In
Logged In


## Step 3: Select Your Project

A **project** in Rhino FCP is the workspace that contains your datasets and defines which sites are participating in the analysis.

Replace `PROJECT_UID` with the exact UID of your project as it appears in the Rhino FCP platform.

In [6]:
PROJECT_UID = "e35dfa01-60cf-4463-8dc9-80585aeb402d"   # Replace with your Project UID

project   = session.project.get_projects(project_uids=[PROJECT_UID])[0]
workgroup = session.project.get_collaborating_workgroups(PROJECT_UID)[0]

print(f"Project \t({project.uid}): \t{project.name}")
print(f"Workgroup \t({workgroup.uid}): \t{workgroup.name}")

Project 	(e35dfa01-60cf-4463-8dc9-80585aeb402d): 	Cox Notebook Test - Julia
Workgroup 	(e590e0fa-ae37-48b3-b50e-c232536cefab): 	Rhino Health


## Step 4: Select Your Datasets

A **dataset** is a table of patient data registered with the platform at a particular site. You'll select the datasets you want to include in this analysis — typically one per participating site.

Replace `DATASET_1` and `DATASET_2` with the exact names of your datasets (should already be registered to the platform). 
Add or remove entries from the list if you have a different number of sites.

In [14]:
dataset_uids = [
    project.get_dataset_by_name("DATASET_1"),
    project.get_dataset_by_name("DATASET_2"),
]
print(f"Dataset 1 Loaded: {dataset_uids[0].name} ({dataset_uids[0].uid})")
print(f"Dataset 2 Loaded: {dataset_uids[1].name} ({dataset_uids[1].uid})")


# NOTE: if demoing internally (Using the Rhino Health Workgroup), register/use the following dummy datasets:
# - /rhino_data/external/import-external-datasets-dev/DATASET_1.csv
# - /rhino_data/external/import-external-datasets-dev/DATASET_2.csv

Dataset 1 Loaded: DATASET_1 (f61401c4-4ab1-4a4f-a8d3-697165297d5a)
Dataset 2 Loaded: DATASET_2 (a15fa9e8-e73e-4b60-a849-ed337e1a081f)


### Expected data format

Each dataset must contain at minimum these columns:

| Column | Description | Example values |
|--------|-------------|---------------|
| `Time` | How long (in any consistent unit, e.g. days) until the event occurred, or until the observation ended | `84.0`, `97.0` |
| `Event` | Whether the event happened: `1` = yes, `0` = no (censored — the observation ended before the event occurred) | `1`, `0` |
| Covariate columns | Any additional variables you want to test as potential influencing factors | `0.3`, `5.3` |

The cell below shows what a small example dataset looks like. Your real datasets will have many more rows.

In [12]:
pd.DataFrame({
    'Time': [84.0, 97.0, 91.0, 90.0, 124.0, 97.0],
    'Event': [1, 0, 0, 1, 1, 1],
    'COV1': [0.3, 0.51, 0.12, 0.03, 0.413, 0.3],
    'COV2': [5.3, 1.51, 1.8, 0.03, 13, 0.3]
})

,Time,Event,COV1,COV2
0,84.0,1,0.300,5.30
1,97.0,0,0.510,1.51
2,91.0,0,0.120,1.80
3,90.0,1,0.030,0.03
4,124.0,1,0.413,13.00
5,97.0,1,0.300,0.30


## Step 5: Configure and Run the Analysis

Now we tell the platform which columns in your data correspond to the time, event, and covariates, then kick off the federated computation.

**Update the three variables below to match your dataset's column names:**

- `time_variable` — the name of the column that records how long until the event (or end of observation)
- `event_variable` — the name of the column that records whether the event occurred (`1`/`0`)
- `covariates` — a list of column names for the factors you want to test

The analysis uses an iterative process (up to `max_iterations=50` rounds) to converge on the best-fit coefficients. The platform coordinates this across all participating sites automatically — no manual steps are needed between rounds.

Once complete, the `results` object will contain the model coefficients, which describe the relationship between each covariate and the timing of the event.

In [13]:
# Set the time and event variables
time_variable = "Time"
event_variable = "Event"
covariates = ["COV1", "COV2"]

# Create a Cox instance, use the mean of the local betas of the two sites as the initial beta
metric_configuration = Cox(time_variable=time_variable, event_variable=event_variable, covariates=covariates, initial_beta="mean", max_iterations=50)

# Retrieve results for your project and datasets
results = project.aggregate_dataset_metric(dataset_uids=[str(dataset.uid) for dataset in dataset_uids], metric_configuration=metric_configuration)

RhinoSDKException: Failed to make request
Status is 500, Trace Id: 0806gccxu8dmgg, Errors: Waiting for connection timed out after 30 seconds., Content is b'{"errors":[{"title":"Error getting aggregate metrics","message":"Waiting for connection timed out after 30 seconds.","extra_info":{}}]}'

